In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("H2_stanza.csv")
df = df.sort_values(by=["id_teks", "id_kalimat", "id"])

In [3]:
# def get_subtree_full(words, head_id):
#     result = []

#     def dfs(node_id):
#         for w in words:
#             if w["head"] == node_id:
#                 dfs(w["id"])
#         for w in words:
#             if w["id"] == node_id:
#                 result.append((w["id"], w["kata"]))

#     dfs(head_id)

#     result = sorted(result, key=lambda x: x[0])
#     return " ".join([r[1] for r in result])

In [4]:
def get_subtree(words, head_id):

    result = []

    def dfs(node_id):

        for w in words:

            if w["head"] == node_id:

                # skip punctuation
                if w["pos"] == "PUNCT":
                    continue

                dfs(w["id"])

        for w in words:

            if w["id"] == node_id:

                # skip punctuation
                if w["pos"] == "PUNCT":
                    continue

                result.append((w["id"], w["kata"]))

    dfs(head_id)

    result = sorted(result, key=lambda x: x[0])

    return " ".join([w[1] for w in result])

In [5]:
# def get_subtree_unique(words, head_id, used_ids):
#     result = []

#     def dfs(node_id):
#         if node_id in used_ids:
#             return

#         used_ids.add(node_id)

#         for w in words:
#             if w["head"] == node_id:
#                 dfs(w["id"])

#         for w in words:
#             if w["id"] == node_id:
#                 result.append((w["id"], w["kata"]))

#     dfs(head_id)

#     return " ".join([r[1] for r in sorted(result)])

In [6]:
def is_keterangan(words, node_id):

    prep_list = [
        "di", "ke", "dari",
        "kepada", "untuk", "buat", "dengan"
    ]

    for w in words:

        if w["head"] == node_id and w["deprel"] == "case":

            if w["kata"].lower() in prep_list:
                return True

    return False

In [ ]:
def get_object_phrase(words, head_id):
    result = []

    def dfs(node_id):
        for w in words:
            if w["head"] == node_id:

                if w["deprel"].startswith("nmod") and is_keterangan(words, w["id"]):
                    continue

                if w["deprel"] in [
                    "compound",
                    "flat",
                    "amod",
                    "det",
                    "nummod"
                ]:
                    dfs(w["id"])

                # khsus relative clause
                elif w["deprel"] == "acl:relcl":

        
                    for child in words:

    
                        if (
                            child["id"] == w["id"]
                            or is_child_of(words, child["id"], w["id"])
                        ):

                            result.append((child["id"], child["kata"]))

                elif w["deprel"].startswith("nmod"):

                    if not is_keterangan(words, w["id"]):
                        dfs(w["id"])

        for w in words:
            if w["id"] == node_id:
                result.append((w["id"], w["kata"]))

    dfs(head_id)

    result = sorted(result, key=lambda x: x[0])

    final_words = []

    for r in result:
        final_words.append(r[1])

    return " ".join(final_words)

In [ ]:
def get_keterangan_from_nmod(words, ket_existing):

    ket = ""

    for w in words:

        if w["deprel"].startswith("nmod"):

            if is_keterangan(words, w["id"]):

                frase = get_subtree(words, w["id"])
                head_word = w["kata"].lower()

                ket_lower = ket_existing.lower()

                
                if head_word in ket_lower:
                    continue

                ket += " " + frase

    return ket.strip()

In [9]:
def get_ket_subtree(words, head_id):

    result = []

    def dfs(node_id):

        for w in words:

            if w["head"] == node_id:

                #  skip possessive manusia
                if (
                    w["deprel"] == "nmod:poss"
                    and w["pos"] == "PRON"
                ):
                    continue

                dfs(w["id"])

        for w in words:

            if w["id"] == node_id:
                result.append((w["id"], w["kata"]))

    dfs(head_id)

    return " ".join(
        [r[1] for r in sorted(result)]
    )

In [ ]:
def add_ket_phrase(ket, frase):

    frase = frase.strip().lower()

    if frase not in ket.lower():
        ket += " " + frase

    return ket.strip()

In [ ]:
def is_child_of(words, child_id, parent_id):
    current = child_id
    while True:
        parent = next((w for w in words if w["id"] == current), None)
        if not parent or parent["head"] == 0:
            return False
        if parent["head"] == parent_id:
            return True
        current = parent["head"]

In [12]:
def remove_duplicate_words(text):
    words = text.split()
    result = []

    for i, w in enumerate(words):
        if i > 0 and w == words[i-1]:
            continue
        result.append(w)

    return " ".join(result)

In [ ]:
def remove_duplicate_phrases(text):

    words = text.split()

    # cek phrase 2 kata berulang
    i = 0
    result = []

    while i < len(words):

        if (
            i + 3 < len(words)
            and words[i] == words[i+2]
            and words[i+1] == words[i+3]
        ):

            result.extend([words[i], words[i+1]])
            i += 4

        else:
            result.append(words[i])
            i += 1

    return " ".join(result)

In [ ]:
def normalize_predikat_objek(predikat, lemma):

    predikat = predikat.lower()
    lemma = lemma.lower()

    # jika kata berakhiran -kan
    if predikat.endswith("kan"):

        # jangan double
        if not lemma.endswith("kan"):
            return lemma + "kan"

        return lemma


    elif predikat.endswith("i"):

        if not lemma.endswith("i"):
            return lemma + "i"

        return lemma

    return lemma

In [ ]:
def fallback_subject(words, root_id):

    for w in words:
        if w["id"] < root_id:

            # manusia/benda
            if w["pos"] in ["PRON", "PROPN", "NOUN"]:

                
                if w["deprel"] in ["nmod:poss", "nmod"]:

                    return get_subtree(words, w["id"])

    return ""

In [16]:
def classify_keterangan(ket):

    ket = ket.lower()

    waktu_keywords = [
        "pagi", "siang", "malam",
        "kemarin", "besok",
        "setelah", "sebelum", "saat"
    ]

    tempat_keywords = [
        "di", "ke", "dari"
    ]

    jenis = []

    if any(k in ket for k in waktu_keywords):
        jenis.append("waktu")

    if any(k in ket.split() for k in tempat_keywords):
        jenis.append("tempat")

    return jenis

In [ ]:
def split_time_from_subject(subj):

    waktu_patterns = [
        "pagi hari",
        "siang hari",
        "malam hari",
        "sore hari"
    ]

    subj_lower = subj.lower()

    for pola in waktu_patterns:

        if subj_lower.startswith(pola):

            new_subj = subj[len(pola):].strip()

            return pola, new_subj

    return "", subj

In [18]:
def get_xcomp_verb(words, root_id):

    verbs = []

    for w in words:

        if (
            w["head"] == root_id
            and w["deprel"] == "xcomp"
            and w["pos"] == "VERB"
        ):

            verbs.append((w["id"], w["kata"]))

    verbs = sorted(verbs, key=lambda x: x[0])

    return " ".join([v[1] for v in verbs])

In [ ]:
def get_subject_phrase(words, head_id):

    result = []

    def dfs(node_id):

        for w in words:

            if w["head"] == node_id:

            
                if (
                    w["deprel"].startswith("nmod")
                    and is_keterangan(words, w["id"])
                ):
                    continue

                dfs(w["id"])

        for w in words:

            if w["id"] == node_id:
                result.append((w["id"], w["kata"]))

    dfs(head_id)

    return " ".join(
        [r[1] for r in sorted(result)]
    )

In [ ]:
def extract_spok(group):

    words = group.to_dict("records")

    subj, pred, obj, ket, lemma_pred = "", "", "", "", ""

    root_id = None

    for w in words:

        if w["deprel"] == "root":

            root_id = w["id"]

            # cari aux
            aux_words = []

            for x in words:
                if x["head"] == root_id and x["deprel"] == "aux":
                    aux_words.append((x["id"], x["kata"]))

            # gabungkan aux + root
            aux_words = sorted(aux_words, key=lambda z: z[0])

            pred_parts = [a[1] for a in aux_words]
            pred_parts.append(w["kata"])

            pred = " ".join(pred_parts)

            xcomp_verb = get_xcomp_verb(words, root_id)

            if xcomp_verb:
                pred += " " + xcomp_verb

            lemma_pred = w["lemma"]

            break

    for w in words:

        # SUBJEK
        if w["deprel"] == "nsubj" and w["head"] == root_id:
            subj = get_subject_phrase(words, w["id"])

        # OBJEK
        elif w["deprel"] == "obj":

            # skip jika objek milik xcomp verb
            parent = next(
                (x for x in words if x["id"] == w["head"]),
                None
            )

            if (
                parent
                and parent["deprel"] == "xcomp"
                and parent["pos"] == "VERB"
            ):

                obj = get_object_phrase(words, w["id"])

            elif w["head"] == root_id:

                obj = get_object_phrase(words, w["id"])

        # KETERANGAN OBL
        elif w["deprel"] == "obl":
            ket = add_ket_phrase(
                ket,
                get_ket_subtree(words, w["id"])
            )

        elif w["deprel"] == "amod" and w["head"] == root_id:

            # cek apakah ada "dengan"
            ada_dengan = any(
                x["head"] == w["id"]
                and x["kata"].lower() == "dengan"
                for x in words
            )
            if ada_dengan:
                continue

            ket += " " + get_subtree(words, w["id"])

        # XCOMP seperti:
        # "dengan ramah"
        elif w["deprel"] == "xcomp":

            # cari apakah ada kata "dengan"
            ada_dengan = any(
                x["head"] == w["id"]
                and x["kata"].lower() == "dengan"
                for x in words
            )

            if ada_dengan:
                ket += " dengan " + get_subtree(words, w["id"])

        
        elif w["deprel"] == "mark":

            if w["kata"].lower() == "dengan":

                # cari target xcomp/amod
                target = next(
                    (
                        x for x in words
                        if x["head"] == root_id
                        and x["deprel"] in ["amod", "xcomp"]
                    ),
                    None
                )

                if target:

                    frase = "dengan " + target["kata"]

                    if frase.lower() not in ket.lower():
                        ket = add_ket_phrase(ket, frase)

        # ADV + AMOD
        elif w["deprel"] == "advmod":

            if any(
                (
                    x["deprel"] == "amod"
                    and is_child_of(words, w["id"], x["id"])
                )
                for x in words
            ):
                continue

            if any(
                (
                    x["deprel"] == "xcomp"
                    and is_child_of(words, w["id"], x["id"])
                )
                for x in words
            ):
                continue

            ket += " " + get_subtree(words, w["id"])

        # RELATIVE CLAUSE ROOT NOUN
        # "kegiatan yang menyenangkan"
        elif (
            w["deprel"] == "acl:relcl"
            and root_id is not None
        ):

            root_word = next(
                (x for x in words if x["id"] == root_id),
                None
            )

            # jika root adalah NOUN
            if root_word and root_word["pos"] == "NOUN":

                frase = get_subtree(words, w["id"])

                ket = add_ket_phrase(ket, frase)

    ket += " " + get_keterangan_from_nmod(words, ket)
    ket = remove_duplicate_words(ket)
    ket = remove_duplicate_phrases(ket)

    if subj == "":
        subj = fallback_subject(words, root_id)

    # pisahkan waktu dari subjek
    ket_waktu, subj_bersih = split_time_from_subject(subj)

    if ket_waktu:

        subj = subj_bersih

        if ket_waktu not in ket:
            ket = ket_waktu + " " + ket

    return (
        subj.strip(),
        pred.strip(),
        obj.strip(),
        ket.strip(),
        lemma_pred.strip()
    )

In [21]:
def detect_prep(ket):
    ket = ket.lower().split()

    for w in ket:
        if w in ["di", "ke", "dari"]:
            return w
    return None

In [ ]:
def is_human_subject(words, subj):

    subj = subj.strip().lower()

    subj_tokens = subj.split()

    subject_words = []

    # ambil token yang termasuk subjek
    for w in words:

        kata = str(w["kata"]).strip().lower()

        if kata in subj_tokens:
            subject_words.append(w)

    # HAPUS TOKEN PENGHUBUNG
    # seperti: dan, atau
 

    filtered_subject_words = []

    for w in subject_words:

        if w["pos"] not in ["CCONJ", "SCONJ", "PUNCT"]:
            filtered_subject_words.append(w)

    # JIKA SUBJEK LEBIH DARI 1 TOKEN
    # DAN ADA TOKEN SELAIN PROPN
    # MAKA ANGGAP BENDA
    if len(filtered_subject_words) > 1:

        for w in filtered_subject_words:

            if w["pos"] not in ["PROPN", "PRON"]:
                return False

    # JIKA SEMUA TOKEN PROPN
    # MAKA ANGGAP MANUSIA
    if len(filtered_subject_words) > 0:

        if all(
            w["pos"] in ["PROPN", "PRON"]
            for w in filtered_subject_words
        ):
            return True

    return False

In [23]:
# =========================================
# DETEKSI OBJEK MANUSIA
# BERDASARKAN HURUF KAPITAL
# =========================================

def is_human_object(obj):

    obj = obj.strip()

    if obj == "":
        return False

    tokens = obj.split()

    for token in tokens:

        # abaikan token pendek
        if len(token) < 2:
            continue

        # jika huruf awal kapital
        if token[0].isupper():
            return True

    return False

In [ ]:
def generate_questions(words, subj, pred, obj, ket, lemma_pred):

    pred_objek = normalize_predikat_objek(pred, lemma_pred)
    questions = []

    # tentukan kata tanya subjek
    tanya_subjek = (
        "Siapa"
        if is_human_subject(words, subj)
        else "Apa"
    )

    if subj and pred and obj and ket:
        questions.append(f"{tanya_subjek} yang {pred} {obj} {ket}?")
        # tentukan kata tanya objek
        tanya_objek = "Siapa" if is_human_object(obj) else "Apa"

        questions.append(
            f"{tanya_objek} yang {subj} {pred_objek} {ket}?"
        )

    elif subj and pred and obj:
        questions.append(f"{tanya_subjek} yang {pred} {obj}?")
        # tentukan kata tanya objek
        tanya_objek = "Siapa" if is_human_object(obj) else "Apa"

        questions.append(
            f"{tanya_objek} yang {subj} {pred_objek}?"
        )

    elif subj and pred:
        questions.append(
            f"{tanya_subjek} yang {pred} {ket}?" if ket else f"{tanya_subjek} yang {pred}?"
        )

    # deteksi waktu berdasarkan kata keterangan
    # if any(k in ket.lower() for k in ["pagi", "siang", "malam", "kemarin"]):
    #     questions.append(f"Kapan {subj} {pred} {obj if obj else ''}?")

    #deteksi tempat berdasarkan preposisi
    jenis_ket = classify_keterangan(ket)

    # pertanyaan waktu berdasarkan kata keterangan
    if "waktu" in jenis_ket:
        questions.append(
            f"Kapan {subj} {pred} {obj if obj else ''}?"
        )

    #pertanyaan tempat berdasarkan preposisi
    prep = detect_prep(ket)

    if "tempat" in jenis_ket:

        if prep == "di":
            questions.append(
                f"Di mana {subj} {pred} {obj if obj else ''}?"
            )

        elif prep == "ke":
            questions.append(
                f"Ke mana {subj} {pred} {obj if obj else ''}?"
            )

        elif prep == "dari":
            questions.append(
                f"Dari mana {subj} {pred} {obj if obj else ''}?"
            )

    # hapus duplicate question
    questions = list(dict.fromkeys(questions))

    return questions

In [25]:
hasil = []

grouped = df.groupby(["id_teks", "id_kalimat"])

for (id_teks, id_kalimat), group in grouped:

    kalimat = group["kalimat"].iloc[0]

    subj, pred, obj, ket, lemma_pred = extract_spok(group)
    questions = generate_questions(
        words=group.to_dict("records"),
        subj=subj,
        pred=pred,
        obj=obj,
        ket=ket,
        lemma_pred=lemma_pred
    )

    for q in questions:
        hasil.append({
            "id_teks": id_teks,
            "id_kalimat": id_kalimat,
            "kalimat": kalimat,
            "subjek": subj,
            "predikat": pred,
            "objek": obj,
            "keterangan": ket,
            "pertanyaan": q
        })

df_final = pd.DataFrame(hasil)
df_final.to_csv("H3_rulebased.csv", index=False)

print("Selesai!")

Selesai!
